# 세그멘테이션 실험 — 레이 특화 yolov8n-seg / Kaggle T4

로컬 20분 실험(seg_v1)의 클라우드 완주 버전. 레이 121장 폴리곤 라벨로 100 epoch.

**실행 전**
- Settings → GPU T4, Internet On
- Add Input: `ray_seg.bin` 이 들어있는 데이터셋
- 학습 후: `/kaggle/working/runs/seg_full/weights/best.pt` 다운로드
  → 로컬 `seg_experiment/runs/seg_full_best.pt` 로 저장

In [ ]:
# 0. 입력 탐색 + 압축 해제
from pathlib import Path
import zipfile, shutil

INPUT_ROOT = Path("/kaggle/input")
hits = list(INPUT_ROOT.rglob("ray_seg.bin"))
assert hits, "ray_seg.bin 못 찾음 — Add Input 확인"
BIN = hits[0]
print("발견:", BIN)

DATA = Path("/kaggle/working/ray_seg")
if DATA.exists():
    shutil.rmtree(DATA)
with zipfile.ZipFile(BIN) as z:
    z.extractall(DATA)
print("해제 완료:", sum(1 for _ in DATA.rglob("*.jpg")), "이미지")

In [ ]:
# 1. data.yaml 경로 재작성 (bin 안에는 로컬 맥 절대경로가 들어있음)
NAMES = ["car_emblem", "door_handle", "fuel_cap", "license_plate",
         "side_mirror", "tail_light"]
yaml_text = (
    f"train: {DATA/'train'/'images'}\n"
    f"val: {DATA/'valid'/'images'}\n\n"
    f"nc: {len(NAMES)}\nnames: {NAMES}\n"
)
(DATA / "data.yaml").write_text(yaml_text)
print(yaml_text)

In [ ]:
# 2. 학습
%pip install -q ultralytics
from ultralytics import YOLO

model = YOLO("yolov8n-seg.pt")
model.train(
    data=str(DATA / "data.yaml"),
    epochs=100, patience=25,
    imgsz=640, batch=32,
    project="/kaggle/working/runs", name="seg_full",
    exist_ok=True, seed=42,
)

In [ ]:
# 3. 결과 확인 — best.pt 를 다운로드해서 seg_experiment/runs/seg_full_best.pt 로
from pathlib import Path
best = Path("/kaggle/working/runs/seg_full/weights/best.pt")
print("best.pt:", best.exists(), best.stat().st_size if best.exists() else "-")